# BioBERT: SDGi Report-Grouped Multilingual Baseline



## 1. Import libraries and set paths


In [3]:
import os
import torch
import json
import numpy as np
import pandas as pd
from IPython.display import display
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
)

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
SEED = 42
set_seed(SEED)

RUN_DIR = "/content/Thesis/base-lines-sdgi-report-grouped/all/biobert"
os.makedirs(RUN_DIR, exist_ok=True)

## 2. Load the dataset


In [5]:
train_df = pd.read_parquet("/content/Thesis/sdgi_report_grouped/data/train.parquet")
validation_df = pd.read_parquet("/content/Thesis/sdgi_report_grouped/data/validation.parquet")
test_df = pd.read_parquet("/content/Thesis/sdgi_report_grouped/data/test.parquet")

frames = {"train": train_df, "validation": validation_df, "test": test_df}
for name, frame in frames.items():
    print(f"{name}: {len(frame):,} passages, {frame['report_id'].nunique():,} reports")
    assert frame["split"].eq(name).all()
assert set(train_df["report_id"]).isdisjoint(validation_df["report_id"])
assert set(train_df["report_id"]).isdisjoint(test_df["report_id"])
assert set(validation_df["report_id"]).isdisjoint(test_df["report_id"])

display(train_df[["text", "labels"]].head())

train: 5,880 passages, 294 reports
validation: 735 passages, 71 reports
test: 735 passages, 71 reports


,text,labels
0,END POVERTY IN ALL ITS FORMS EVERYWHERE 22 Pov...,[1]
1,"15 With regard to this SDG in particular, it i...",[1]
2,"ࠀ߿ |Fin de la pobreza ࠀ.ࠃ De aquí a ߿ࠂ߿ࠁ, gara...",[1]
3,22 melbourne.vic.gov.au Context Poverty is a c...,[1]
4,20 SDG 1 – End poverty in all its forms everyw...,[1]


## 2.1. Connect to W&B

.env file supplies WANDB_API_KEY and HF_TOKEN.


In [7]:
import torch
import wandb

WANDB_PROJECT = "thesis-sdgi-report-grouped"
WANDB_ENTITY = None
RUN_NAME = "biobert/all"

# I close the previous run before starting a new one.
wandb.finish()
os.environ["WANDB_NOTEBOOK_NAME"] = f"{RUN_DIR}/biobert_finetuned.ipynb"
with open("/content/Thesis/.env") as env_file:
    for line in env_file:
        name, separator, value = line.strip().removeprefix("export ").partition("=")
        if separator and name.strip() in ["WANDB_API_KEY", "HF_TOKEN"]:
            os.environ[name.strip()] = value.strip().strip("\"'")
if not os.environ.get("WANDB_API_KEY"):
    raise ValueError("WANDB_API_KEY is missing from the thesis .env file.")
wandb.login()
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    group="all",
    job_type="biobert",
    tags=["all", "biobert", "report-grouped", "samples-metrics"],
    dir=RUN_DIR,
    config={
        "checkpoint": MODEL_NAME, "dataset": "sdgi_report_grouped",
        "gpu": torch.cuda.get_device_name(0), "seed": SEED, "epochs": 10, "learning_rate": 2e-5,
        "batch_size": 4, "weight_decay": 0.01, "max_length": 512,
        "threshold": 0.5, "metrics_average": "samples", "checkpoint_selection": "validation_loss",
        "train_passages": len(train_df), "validation_passages": len(validation_df),
        "test_passages": len(test_df),
    },
)
print("W&B run:", run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: amansharma0603 (amansharma0603-university-of-new-south-whales) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/amansharma0603-university-of-new-south-whales/thesis-sdgi-report-grouped/runs/1p9xkke8


## 3. Encode SDG labels and tokenize passages


In [9]:
def make_dataset(frame):
    labels = [[float(goal in goals) for goal in range(1, 18)] for goals in frame["labels"]]
    return Dataset.from_dict({"text": frame["text"].tolist(), "labels": labels})

# I use the original tokenizer for each checkpoint.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, token=os.environ.get("HF_TOKEN") or False)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

train_data = make_dataset(train_df).map(tokenize, batched=True, remove_columns=["text"])
validation_data = make_dataset(validation_df).map(tokenize, batched=True, remove_columns=["text"])
test_data = make_dataset(test_df).map(tokenize, batched=True, remove_columns=["text"])

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5880 [00:00<?, ? examples/s]

Map:   0%|          | 0/735 [00:00<?, ? examples/s]

Map:   0%|          | 0/735 [00:00<?, ? examples/s]

## 4. Load the model and define metrics

Predictions use a threshold of 0.5. Precision, recall and F1 are averaged across passages; accuracy requires an exact label-set match.


In [11]:
id2label = {index: f"SDG {index + 1}" for index in range(17)}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    token=os.environ.get("HF_TOKEN") or False,
    num_labels=17,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id={label: index for index, label in id2label.items()}
)

def compute_metrics(prediction):
    probabilities = torch.sigmoid(torch.tensor(prediction.predictions)).numpy()
    predicted_labels = (probabilities >= 0.5).astype(int)
    true_labels = prediction.label_ids

    return {
        "precision": precision_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "recall": recall_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "f1": f1_score(true_labels, predicted_labels, average="samples", zero_division=0)
    }


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Train for 10 epochs


In [13]:
training_args = TrainingArguments(
    output_dir=f"{RUN_DIR}/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="wandb",
    run_name=RUN_NAME,
    logging_strategy="epoch",
    dataloader_pin_memory=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,0.199500,0.093335,0.883447,0.864390,0.851701,0.868621
2,0.096300,0.078953,0.911995,0.898974,0.869388,0.897647
3,0.066600,0.072111,0.922530,0.914183,0.878912,0.910052
4,0.049100,0.066787,0.930570,0.921676,0.887075,0.919672
5,0.037200,0.062545,0.940181,0.926942,0.893878,0.926695
6,0.028600,0.058635,0.942802,0.936550,0.897959,0.933321
7,0.022000,0.063174,0.939940,0.931596,0.891156,0.928895
8,0.016400,0.064016,0.941100,0.933951,0.893878,0.930982
9,0.013000,0.066795,0.938698,0.928508,0.892517,0.926703
10,0.010900,0.066742,0.939423,0.927294,0.893878,0.926312


TrainOutput(global_step=14700, training_loss=0.05395937368172367, metrics={'train_runtime': 656.5736, 'train_samples_per_second': 89.556, 'train_steps_per_second': 22.389, 'total_flos': 1.546611223585428e+16, 'train_loss': 0.05395937368172367, 'epoch': 10.0})

## 6. Evaluate and save


In [15]:
import torch
validation_results = trainer.evaluate(validation_data, metric_key_prefix="validation")
test_results = trainer.evaluate(test_data, metric_key_prefix="test")

trainer.save_model(f"{RUN_DIR}/model")
tokenizer.save_pretrained(f"{RUN_DIR}/model")

results = {
    "model": "BioBERT", "checkpoint": MODEL_NAME, "dataset": "sdgi_report_grouped",
    "gpu": torch.cuda.get_device_name(0), "seed": SEED, "epochs": 10, "learning_rate": 2e-5,
    "batch_size": 4, "max_length": 512, "threshold": 0.5, "metrics_average": "samples",
    "train_passages": len(train_df), "validation_passages": len(validation_df),
    "test_passages": len(test_df),
    **validation_results, **test_results,
}
results_df = pd.DataFrame([results])
results_df.to_csv(f"{RUN_DIR}/results.csv", index=False)
with open(f"{RUN_DIR}/training_history.json", "w") as file:
    json.dump(trainer.state.log_history, file, indent=2)
display(results_df[["model", "validation_loss", "validation_f1",
                    "test_precision", "test_recall", "test_f1", "test_accuracy"]])
print("Model and results saved to:", RUN_DIR)
# I add the final scores to the W&B summary.
run.summary["gpu"] = torch.cuda.get_device_name(0)
run.summary.update(validation_results)
run.summary.update(test_results)
run.summary["best_validation_loss"] = trainer.state.best_metric
run.summary["completed_epochs"] = trainer.state.epoch
run.summary["local_model_directory"] = f"{RUN_DIR}/model"
print("W&B run:", run.url)
run.finish()

,model,validation_loss,validation_f1,test_precision,test_recall,test_f1,test_accuracy
0,BioBERT,0.058635,0.933321,0.909349,0.884472,0.882616,0.797279


Model and results saved to: /content/Thesis/base-lines-sdgi-report-grouped/all/biobert
W&B run: https://wandb.ai/amansharma0603-university-of-new-south-whales/thesis-sdgi-report-grouped/runs/1p9xkke8


eval/accuracy,▁▄▅▆▇█▇▇▇▇
eval/f1,▁▄▅▇▇███▇▇
eval/loss,█▅▄▃▂▁▂▂▃▃
eval/precision,▁▄▆▇██████
eval/recall,▁▄▆▇▇███▇▇
eval/runtime,▂▆▂▆▂▅▂▃▁█
eval/samples_per_second,▇▃▇▂▇▄▇▆█▁
eval/steps_per_second,▇▃▇▂▇▄▇▆█▁
test/accuracy,▁
test/f1,▁
+19,...
